# Respiratory incidence ML panel

This notebook builds a larger supervised modelling panel from the predictive and predicted respiratory-virus data sources in the repository.

It uses all available predictive series, creates lagged features, and evaluates several model classes under a chronological train-test split. For a fully scripted run, use `python scripts/run_respiratory_ml_pipeline.py`.

## Data included

Predictive families currently gathered automatically:

- Google Trends one-year respiratory search terms
- raw wastewater data from `data/raw` using `sources.csv`
- UKHSA NHS-call chart exports
- optional processed wastewater panels
- downloaded OWID COVID surveillance series
- downloaded Open-Meteo weather series

Predicted families currently gathered automatically:

- UKHSA GP/admission chart exports
- downloaded OWID COVID incidence / severity targets

Additional catalogued external sources are listed in `predictive_sources.csv` and `predicted_sources.csv`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from wastewater.io import find_repo_root
from wastewater.external_series import build_all_available_series
from wastewater.regression_matrix import summarise_available_series
from wastewater.ml_panel import PanelBuildConfig, build_lagged_feature_panel, evaluate_all_targets

ROOT = find_repo_root(ROOT)
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
ROOT

## 1. Review source catalogues

In [ ]:
predictive_sources = pd.read_csv(ROOT / 'predictive_sources.csv')
predicted_sources = pd.read_csv(ROOT / 'predicted_sources.csv')

display(predictive_sources)
display(predicted_sources)

## 2. Gather available canonical series

This loads local repo data plus any downloaded external series in `data/external`. To fetch OWID COVID and Open-Meteo weather first, run:

```bash
python scripts/download_external_respiratory_sources.py
```

In [ ]:
series = build_all_available_series(ROOT, include_external=True)
print(f'Canonical observations: {len(series):,}')

series_summary = summarise_available_series(series)
display(series_summary)

if not series.empty:
    display(series.groupby(['role', 'dataset_family'])['series_id'].nunique().reset_index(name='n_series'))

series.to_csv(PROCESSED / 'respiratory_ml_canonical_series.csv', index=False)

## 3. Configure lagged panel

The default uses weekly data and lags predictive variables by 1--8 weeks. This avoids using same-week information as the only signal and gives the model lead-time features.

In [ ]:
CONFIG = PanelBuildConfig(
    freq='W',
    lags=(1, 2, 3, 4, 5, 6, 7, 8),
    aggregation='mean',
    min_non_missing_fraction=0.2,
)
TRAIN_FRACTION = 0.8
MIN_TEST_SIZE = 4
CONFIG

## 4. Inspect one target panel

This sanity-checks the feature matrix size before fitting all targets.

In [ ]:
target_ids = sorted(series.loc[series['role'] == 'predicted', 'series_id'].dropna().unique())
print(f'Predicted targets: {len(target_ids)}')
for target_id in target_ids[:10]:
    print(' -', target_id)

if target_ids:
    target_id = target_ids[0]
    panel, feature_cols = build_lagged_feature_panel(series, target_id=target_id, config=CONFIG)
    print('Example target:', target_id)
    print('Panel rows:', len(panel))
    print('Feature columns:', len(feature_cols))
    display(panel.head())

## 5. Evaluate models for every predicted target

Models currently compared:

- OLS
- Ridge regression
- Elastic net
- Random forest
- Histogram gradient boosting

All use the same chronological train-test split.

In [ ]:
ml_results, ml_predictions = evaluate_all_targets(
    series,
    config=CONFIG,
    train_fraction=TRAIN_FRACTION,
    min_test_size=MIN_TEST_SIZE,
    random_state=42,
)

display(ml_results)

results_path = PROCESSED / 'respiratory_incidence_ml_model_results.csv'
predictions_path = PROCESSED / 'respiratory_incidence_ml_model_predictions.csv'
ml_results.to_csv(results_path, index=False)
ml_predictions.to_csv(predictions_path, index=False)
print(f'Saved model results to {results_path.relative_to(ROOT)}')
print(f'Saved model predictions to {predictions_path.relative_to(ROOT)}')

## 6. Rank models by held-out skill

The key metric is `mse_skill_vs_train_mean`. Positive values mean the model improves on the training-mean baseline in the held-out test set.

In [ ]:
ok = ml_results[ml_results['status'] == 'ok'].copy() if not ml_results.empty and 'status' in ml_results.columns else pd.DataFrame()
if ok.empty:
    print('No successful model fits. Inspect error rows.')
    display(ml_results)
else:
    rank_cols = ['target_id', 'model', 'n_train', 'n_test', 'n_features', 'rmse', 'baseline_rmse', 'mse_skill_vs_train_mean', 'correlation', 'r2']
    display(ok[rank_cols].sort_values('mse_skill_vs_train_mean', ascending=False).head(50))

## 7. Plot the best model

In [ ]:
if not ok.empty and not ml_predictions.empty:
    best = ok.sort_values('mse_skill_vs_train_mean', ascending=False).iloc[0]
    pred = ml_predictions[
        (ml_predictions['target_id'] == best['target_id']) &
        (ml_predictions['model'] == best['model'])
    ].copy()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(pred['period'], pred['target'], label='Observed test', marker='o')
    ax.plot(pred['period'], pred['prediction'], label='Predicted test', marker='o')
    ax.plot(pred['period'], pred['baseline_prediction'], label='Training-mean baseline', linestyle='--')
    ax.set_title('Best held-out ML model: ' + str(best['model']) + ' → ' + str(best['target_id']))
    ax.set_xlabel('Period')
    ax.set_ylabel('Target value')
    ax.legend()
    fig.autofmt_xdate()
    plt.show()

    display(best.to_frame('value'))
    display(pred)

## 8. Next steps

- Add robust downloaders/parsers for the remaining catalogued sources.
- Use a multi-region panel once each source has geographic identifiers.
- Replace a single train/test split with rolling-origin validation.
- Add count models or Poisson/Tweedie objectives for raw count targets.
- Use feature importance or SHAP for model interpretation once performance is stable.